<a href="https://colab.research.google.com/github/heisenberg304/heart-disease-ml-classification/blob/main/RandomForest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [53]:
from model_1 import model_binary, predict_modelo_1 as predict_1      #binaria
from model_2 import model_multiclass, predict_modelo_2 as predict_2  #multiclase
from data_engineering import data, FEATURES, split_dataset

from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

new_patients = data.sample()

#por ahora usaremos el x_test y y_test de data, pero data puede ser reemplazada por una nueva dataset de pacientes
def optional_data(data):
  x = data[FEATURES]
  y = data["num"]
  x_train, x_test, y_train, y_test = split_dataset(x, y)
  return x_test, y_test
x_test, y_test = optional_data(data)

#pacientes totalmente nuevos, solo features
def pred_binaria(new_patients):
  x = new_patients[FEATURES]   #como es nuevo no tiene la columna target ni nada solo los Features, tambien sirve por si en los datos nuevos hay otras columnas innecesarias como id o name
  pred_1 = predict_modelo_1(model_binary, x)
  return pred_1
pred_1 = pred_binaria(new_patients)
print(pred_1)



[1]


In [36]:
%%writefile data_engineering.py

#Cargar librerias
import pandas as pd
from sklearn.model_selection import train_test_split


#Cargar dataset
data = pd.read_csv("/content/drive/MyDrive/data_csv/heart_disease_uci.csv")

#===================================================================================================================================================================================================

#MANIPULAR CSV
  #rellenar: trestbps(59)(median o mean), chol(30)(median), fbs(90)(mode), restecg(2)(mode), thalch(55)(median o mean), oldpeak(62)(median)
def handle_missing_values(data):
  data["trestbps"].fillna(data["trestbps"].median(), inplace=True)
  data["chol"].fillna(data["chol"].median(), inplace=True)
  data["fbs"].fillna(data["fbs"].mode()[0], inplace=True)
  data["restecg"].fillna(data["restecg"].mode()[0], inplace=True)
  data["thalch"].fillna(data["thalch"].median(), inplace=True)
  data["oldpeak"].fillna(data["oldpeak"].median(), inplace=True)
  return data

data = handle_missing_values(data)

def encode_categorical_features(data):
    #encoding: sex, cp, restecg, num (target)
  fe_ma = pd.get_dummies(data["sex"])
  CP = pd.get_dummies(data["cp"])
  RESTECG = pd.get_dummies(data["restecg"])
  return fe_ma, CP, RESTECG

fe_ma, CP, RESTECG = encode_categorical_features(data)


def feature_selection(data, fe_ma, CP, RESTECG):
    #union de datasets
  data = pd.concat([data, fe_ma, CP, RESTECG], axis=1)
    #borrar columnas
  data = data.drop(["id", "dataset", "slope", "ca", "thal", "sex", "cp", "restecg"], axis=1)
  return data

data = feature_selection(data, fe_ma, CP, RESTECG)

#===================================================================================================================================================================================================

FEATURES = ['age','trestbps','chol','fbs','thalch','exang',
            'oldpeak','Female','Male','asymptomatic',
            'atypical angina','non-anginal','typical angina',
            'lv hypertrophy','normal','st-t abnormality']

##############################################################

#modelo 1
def prepare_binary_dataset(data):
  #copiar dataframe
  data = data.copy()
      #target binario - separar enfermos y no enfermos
  data["num"] = data["num"].apply(lambda x: 1 if x > 0 else 0)
  # crear x , y
  x = data[FEATURES]
  y = data["num"]
  return x, y

##############################################################

#modelo 2
def prepare_multiclass_dataset(data):
  #copiar dataframe
  data = data.copy()
      #target multivariable - clasificación de severidad
  data = data[data["num"] > 0]
  # crear x , y
  x = data[FEATURES]
  y = data["num"]
  return x, y

##############################################################

def split_dataset(x, y):
  #split 80/20
  x_train, x_test, y_train, y_test = train_test_split(
      x,
      y,
      test_size = 0.2,
      random_state=55
  )
  return x_train, x_test, y_train, y_test


Writing data_engineering.py


In [37]:
%%writefile model_1.py

#funciones de data_engineering
from data_engineering import split_dataset, prepare_binary_dataset, data

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report


x, y = prepare_binary_dataset(data)
x_train, x_test, y_train, y_test = split_dataset(x, y)

def modelo_1():
  #crear modelo
  model = RandomForestClassifier(class_weight = "balanced",
                                n_estimators=200,
                                max_depth=7,
                                min_samples_leaf=6,
                                random_state=7,
                                max_features="sqrt")
  return model

model_binary = modelo_1()

def entrenar_modelo_1(model_binary, x_train, y_train):
  #entrenar modelo
  model_binary.fit(x_train, y_train)
  return model_binary

model_binary = entrenar_modelo_1(model_binary, x_train, y_train)

def predict_modelo_1(model_binary, x_test):
  #prediccion
  probs = model_binary.predict_proba(x_test)[:, 1]
  threshold = 0.34
  pred = (probs >= threshold).astype(int)
  return pred

pred = predict_modelo_1(model_binary, x_test)

def importances_modelo_1(model_binary):
  #Importancia de cada feature
  importances = model_binary.feature_importances_
  return importances

importances = importances_modelo_1(model_binary)

#===================================================================================================================================================================================================

#metricas
matrix = confusion_matrix(y_test, pred)
todo = classification_report(y_test, pred)

#print de metricas
print(f"matrix -> {matrix}")
print(f"todo -> {todo}")
print(f"Importancia = {importances}")




Writing model_1.py



#==========================================================================================================================================

#MODEL PERFORMANCE

# -----confusion matrix--------
# |          Pred 0   Pred 1  |
# | Real 0     56       28    |
# | Real 1      7       93    |
# -----------------------------
#accuracy  -> 0.81
#precision -> 0.77
#recall    -> 0.93
#f1 score  -> 0.84
#Analisis:
#El modelo logra un recall alto , lo cual indica que detecta correctamente la mayoría de pacientes con enfermedad.
#Sin embargo, la precisión es menor , lo que implica falsos positivos.
#Este comportamiento es aceptable dado que en un contexto médico es preferible no dejar pasar casos enfermos.

#Los 3 Importance mas relevantes
#"asymptomatic" [0.22129586] El modelo detecta que la ausencia de síntomas típicos no implica ausencia de enfermedad, lo cual es clínicamente coherente.
#"oldpeak"      [0.12007416] Es una variable médica directa relacionada con problemas cardíacos, por lo que tiene fuerte poder predictivo.
#"exang"        [0.11460541] Indica que el corazón no responde bien al esfuerzo, lo que es un fuerte indicador clínico




In [40]:
%%writefile model_2.py


#funciones de data_engineering
from data_engineering import split_dataset, prepare_multiclass_dataset, data

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

x, y = prepare_multiclass_dataset(data)
x_train, x_test, y_train, y_test = split_dataset(x, y)


def modelo_2():
  #crear modelo
  model = RandomForestClassifier(class_weight = "balanced",
                                n_estimators=200,
                                max_depth=9,
                                min_samples_leaf=6,
                                random_state=1,
                                max_features="sqrt")
  return model

model_multiclass = modelo_2()

def entrenar_modelo_2(model_multiclass, x_train, y_train):
  #entrenar modelo
  model_multiclass.fit(x_train, y_train)
  return model_multiclass

model_multiclass = entrenar_modelo_2(model_multiclass, x_train, y_train)

def predict_modelo_2(model_multiclass, x_test):
  #prediccion
  pred = model_multiclass.predict(x_test)
  return pred

pred = predict_modelo_2(model_multiclass, x_test)

def importances_modelo_2(model_multiclass):
  #Importancia de cada feature
  importances = model_multiclass.feature_importances_
  return importances

importances = importances_modelo_2(model_multiclass)

#===================================================================================================================================================================================================

#metricas
matrix = confusion_matrix(y_test, pred)
todo = classification_report(y_test, pred)

#print de metricas
print(f"matrix -> {matrix}")
print(f"todo -> {todo}")
print(f"Importancia = {importances}")


Writing model_2.py
